# FER2013 Facial Expression Recognition
## Assignment 4 — Machine Learning 2026

### Setup order:
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Run **Section 1** (install + clone repo)
3. Run **Section 2** (Kaggle + download data)
4. Run **Section 3** (Wandb login)
5. Run **Section 4** (train all experiments)


## Section 1 — Install dependencies & clone repo

In [6]:
# Install all required packages
!pip install -q torch torchvision wandb kaggle tqdm seaborn scikit-learn

# Clone YOUR GitHub repo (replace with your actual repo URL after creating it)
# REPO_URL = "https://github.com/YOUR_USERNAME/fer2013-emotion"
# !git clone $REPO_URL
# %cd fer2013-emotion

# For now, just create the folder structure directly
import os
os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
print('✅ Setup complete')

✅ Setup complete


## Section 2 — Download FER2013 from Kaggle

In [7]:
# Option A: Upload kaggle.json manually
# Go to kaggle.com → Account → Create New API Token → download kaggle.json
from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()  # select kaggle.json when prompted

import os, json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(json.loads(list(uploaded.values())[0]), f)

os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle credentials saved')

Upload your kaggle.json file:


Saving kaggle.json to kaggle.json
✅ Kaggle credentials saved


In [8]:
# Download and unzip the FER2013 dataset
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge -p data/
!unzip -q data/challenges-in-representation-learning-facial-expression-recognition-challenge.zip -d data/
!ls data/
print('✅ Dataset downloaded')

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
unzip:  cannot find or open data/challenges-in-representation-learning-facial-expression-recognition-challenge.zip, data/challenges-in-representation-learning-facial-expression-recognition-challenge.zip.zip or data/challenges-in-representation-learning-facial-expression-recognition-challenge.zip.ZIP.
fer2013.zip  test  test.csv  train  train.csv
✅ Dataset downloaded


## Section 3 — Login to Weights & Biases

In [9]:
import wandb
# This will prompt you to paste your API key from wandb.ai/authorize
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sdeli23 (sdeli23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Section 4 — Copy source files (if not using git clone)

In [10]:
# If you cloned your repo, skip this cell.
# Otherwise, upload the src/ files manually:
from google.colab import files
print('Upload all files from the src/ folder (data.py, models.py, sanity_checks.py, utils.py, train.py):')
uploaded = files.upload()

import os
os.makedirs('src', exist_ok=True)
for fname, content in uploaded.items():
    with open(f'src/{fname}', 'wb') as f:
        f.write(content)
    print(f'  saved src/{fname}')
print('✅ Source files ready')

Upload all files from the src/ folder (data.py, models.py, sanity_checks.py, utils.py, train.py):


Saving data.py to data.py
Saving models.py to models.py
Saving sanity_checks.py to sanity_checks.py
Saving train.py to train.py
Saving utils.py to utils.py
  saved src/data.py
  saved src/models.py
  saved src/sanity_checks.py
  saved src/train.py
  saved src/utils.py
✅ Source files ready


## Section 5 — Quick sanity check before training

In [11]:
import sys
sys.path.insert(0, 'src')

import torch
from models import get_model, count_parameters
from sanity_checks import run_all_sanity_checks

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Test all three architectures
for arch in ['shallow', 'medium', 'resnet']:
    model = get_model(arch).to(device)
    print(f'\n{arch}: {count_parameters(model):,} parameters')
    run_all_sanity_checks(model, device, arch, log_to_wandb=False)

Device: cuda

shallow: 595,655 parameters

  SANITY CHECKS for SHALLOW
  [shape check] expected (8, 7), got torch.Size([8, 7])  ✅ PASS
  [init loss]   expected ≈1.946, got 1.968  ✅ PASS
  [tiny-batch]  loss after 100 steps: 0.0017  ✅ PASS
  [grad norm]   step 1: 4.6605  ✅
  [grad norm]   step 2: 6.6789  ✅
  [grad norm]   step 3: 5.1328  ✅
  [grad norm]   step 4: 4.1787  ✅
  [grad norm]   step 5: 3.0176  ✅

  Overall: ✅ ALL PASSED


medium: 1,469,031 parameters

  SANITY CHECKS for MEDIUM
  [shape check] expected (8, 7), got torch.Size([8, 7])  ✅ PASS
  [init loss]   expected ≈1.946, got 1.945  ✅ PASS
  [tiny-batch]  loss after 100 steps: 0.0128  ✅ PASS
  [grad norm]   step 1: 15.2903  ✅
  [grad norm]   step 2: 37.8751  ✅
  [grad norm]   step 3: 28.8701  ✅
  [grad norm]   step 4: 31.8925  ✅
  [grad norm]   step 5: 22.7126  ✅

  Overall: ✅ ALL PASSED


resnet: 3,144,391 parameters

  SANITY CHECKS for RESNET
  [shape check] expected (8, 7), got torch.Size([8, 7])  ✅ PASS
  [init loss]   

## Section 6 — Run Architecture 1: ShallowCNN

In [12]:
WANDB_PROJECT = 'fer2013-emotion'   # change to your project name
WANDB_ENTITY  = ''                  # your wandb username (or leave empty)
CSV_PATH      = 'data/train.csv'

entity_flag = f'--wandb_entity {WANDB_ENTITY}' if WANDB_ENTITY else ''
base = f'python src/train.py --csv {CSV_PATH} --wandb_project {WANDB_PROJECT} {entity_flag}'

# Best config
!{base} --arch shallow --epochs 40 --lr 1e-3 --dropout 0.3 --batch_size 64 --run_name shallow_best

[train] device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sdeli23 (sdeli23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run xquhv0mc (0.0s)
wandb: ⣻ setting up run xquhv0mc (0.0s)
wandb: ⣽ setting up run xquhv0mc (0.0s)
wandb: ⣾ setting up run xquhv0mc (0.0s)
wandb: ⣷ setting up run xquhv0mc (0.5s)
wandb: ⣯ setting up run xquhv0mc (0.5s)
wandb: ⣟ setting up run xquhv0mc (0.5s)
wandb: ⡿ setting up run xquhv0mc (0.5s)
wandb: ⢿ setting up run xquhv0mc (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/wandb/run-20260617_192102-xquhv0mc
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run shallow_best
wandb: ⭐️ View project at https://wandb.ai/sdeli23-free-university-of-tbilisi-/fer2013-emotion
wandb: 🚀 View run at h

In [13]:
# Intentional OVERFIT (no dropout, high LR) — required by rubric
!{base} --arch shallow --epochs 40 --lr 5e-3 --dropout 0.0 --weight_decay 0.0 --run_name shallow_OVERFIT

[train] device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sdeli23 (sdeli23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run ldf4jmiw (0.0s)
wandb: ⣻ setting up run ldf4jmiw (0.0s)
wandb: ⣽ setting up run ldf4jmiw (0.0s)
wandb: ⣾ setting up run ldf4jmiw (0.0s)
wandb: ⣷ setting up run ldf4jmiw (0.5s)
wandb: ⣯ setting up run ldf4jmiw (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/wandb/run-20260617_193556-ldf4jmiw
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run shallow_OVERFIT
wandb: ⭐️ View project at https://wandb.ai/sdeli23-free-university-of-tbilisi-/fer2013-emotion
wandb: 🚀 View run at https://wandb.ai/sdeli23-free-university-of-tbilisi-/fer2013-emotion/runs/ldf4jmiw
[data] train=25838  val=2871
[data]

In [2]:
# Intentional UNDERFIT (too much dropout, tiny LR) — required by rubric
!{base} --arch shallow --epochs 40 --lr 1e-4 --dropout 0.7 --run_name shallow_UNDERFIT

/bin/bash: line 1: {base}: command not found


## Section 7 — Run Architecture 2: MediumCNN

In [ ]:
# Best config
!{base} --arch medium --epochs 60 --lr 3e-4 --dropout 0.25 --weight_decay 1e-4 --run_name medium_best

In [ ]:
# High LR variant (overfit / unstable)
!{base} --arch medium --epochs 60 --lr 1e-3 --dropout 0.25 --weight_decay 0.0 --run_name medium_highlr

In [ ]:
# More regularisation (underfit territory)
!{base} --arch medium --epochs 60 --lr 1e-4 --dropout 0.5 --weight_decay 1e-3 --run_name medium_heavy_reg

In [ ]:
# Large batch (often underfits with same LR)
!{base} --arch medium --epochs 60 --lr 3e-4 --dropout 0.25 --batch_size 256 --run_name medium_largebatch

## Section 8 — Run Architecture 3: SmallResNet

In [ ]:
# Best config (with label smoothing — helps with noisy FER2013 labels)
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.3 --label_smoothing 0.1 --weight_decay 1e-4 --run_name resnet_best

In [ ]:
# No label smoothing — typically slightly more overfit
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.3 --label_smoothing 0.0 --run_name resnet_no_smoothing

In [ ]:
# High weight decay
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.2 --weight_decay 1e-3 --run_name resnet_high_wd

## Section 9 — Generate Wandb Report

After all runs finish:
1. Go to **wandb.ai** → your project **fer2013-emotion**
2. Click **Reports** → **New Report**
3. Add panels:
   - Line chart: `val_acc` grouped by `arch` (compare all 3 families)
   - Line chart: `train_acc` vs `val_acc` for each architecture (shows overfit/underfit)
   - Bar chart: `best_val_acc` per run
   - Media panel: confusion matrices
4. Add text sections explaining each architecture's decisions
5. Save and share the report link — paste it in your README


In [ ]:
# Optional: Print wandb project URL
print(f'Your wandb project: https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}')
print('Go there now to see all your logged runs and create a Report!')

In [ ]:
import os
os.makedirs('data', exist_ok=True)

!kaggle datasets download -d msambare/fer2013 -p data/
!unzip -q data/fer2013.zip -d data/
!ls data/

In [5]:
import os, pandas as pd, numpy as np
from PIL import Image

def folder_to_csv(split, output_path):
    emotion_map = {'angry':0,'disgust':1,'fear':2,'happy':3,'neutral':6,'sad':4,'surprise':5}
    rows = []
    for emotion_name, label in emotion_map.items():
        folder = f'data/{split}/{emotion_name}'
        if not os.path.exists(folder): continue
        for fname in os.listdir(folder):
            if not fname.endswith(('.jpg','.png')): continue
            img = Image.open(f'{folder}/{fname}').convert('L').resize((48,48))
            pixels = ' '.join(str(p) for p in np.array(img).flatten())
            rows.append({'emotion':label,'pixels':pixels,'Usage':'Training' if split=='train' else 'PublicTest'})
    pd.DataFrame(rows).to_csv(output_path, index=False)
    print(f'✅ {output_path} done')

folder_to_csv('train','data/train.csv')
folder_to_csv('test','data/test.csv')

✅ data/train.csv done
✅ data/test.csv done


In [4]:
from google.colab import files
import os

os.makedirs('src', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

print('Upload all 5 files from your src/ folder:')
uploaded = files.upload()

for fname, content in uploaded.items():
    with open(f'src/{fname}', 'wb') as f:
        f.write(content)
    print(f'✅ saved src/{fname}')

Upload all 5 files from your src/ folder:


Saving data.py to data.py
Saving models.py to models.py
Saving sanity_checks.py to sanity_checks.py
Saving train.py to train.py
Saving utils.py to utils.py
✅ saved src/data.py
✅ saved src/models.py
✅ saved src/sanity_checks.py
✅ saved src/train.py
✅ saved src/utils.py


In [6]:
import os
import sys
sys.path.insert(0, 'src')

os.makedirs('checkpoints', exist_ok=True)

WANDB_PROJECT = 'fer2013-emotion'
WANDB_ENTITY  = ''
CSV_PATH      = 'data/train.csv'

entity_flag = f'--wandb_entity {WANDB_ENTITY}' if WANDB_ENTITY else ''
base = f'python src/train.py --csv {CSV_PATH} --wandb_project {WANDB_PROJECT} {entity_flag}'

!{base} --arch shallow --epochs 40 --lr 1e-4 --dropout 0.7 --run_name shallow_UNDERFIT
!{base} --arch medium --epochs 60 --lr 3e-4 --dropout 0.25 --weight_decay 1e-4 --run_name medium_best
!{base} --arch medium --epochs 60 --lr 1e-3 --dropout 0.25 --weight_decay 0.0 --run_name medium_highlr
!{base} --arch medium --epochs 60 --lr 1e-4 --dropout 0.5 --weight_decay 1e-3 --run_name medium_heavy_reg
!{base} --arch medium --epochs 60 --lr 3e-4 --dropout 0.25 --batch_size 256 --run_name medium_largebatch
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.3 --label_smoothing 0.1 --weight_decay 1e-4 --run_name resnet_best
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.3 --label_smoothing 0.0 --run_name resnet_no_smoothing
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.2 --weight_decay 1e-3 --run_name resnet_high_wd

[train] device: cuda
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 2
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sdeli23 (sdeli23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/wandb/run-20

In [7]:
import os, pandas as pd, numpy as np
from PIL import Image

def folder_to_csv(split, output_path):
    emotion_map = {'angry':0,'disgust':1,'fear':2,'happy':3,'neutral':6,'sad':4,'surprise':5}
    rows = []
    for emotion_name, label in emotion_map.items():
        folder = f'data/{split}/{emotion_name}'
        if not os.path.exists(folder): continue
        for fname in os.listdir(folder):
            if not fname.endswith(('.jpg','.png')): continue
            img = Image.open(f'{folder}/{fname}').convert('L').resize((48,48))
            pixels = ' '.join(str(p) for p in np.array(img).flatten())
            rows.append({'emotion':label,'pixels':pixels,'Usage':'Training' if split=='train' else 'PublicTest'})
    pd.DataFrame(rows).to_csv(output_path, index=False)
    print(f'✅ {output_path} — {len(rows)} rows')

folder_to_csv('train','data/train.csv')
folder_to_csv('test','data/test.csv')

✅ data/train.csv — 0 rows
✅ data/test.csv — 0 rows


In [8]:
import os, sys
sys.path.insert(0, 'src')
os.makedirs('checkpoints', exist_ok=True)

WANDB_PROJECT = 'fer2013-emotion'
WANDB_ENTITY  = ''
CSV_PATH      = 'data/train.csv'
entity_flag   = ''
base = f'python src/train.py --csv {CSV_PATH} --wandb_project {WANDB_PROJECT}'

!{base} --arch shallow --epochs 40 --lr 1e-4 --dropout 0.7 --run_name shallow_UNDERFIT
!{base} --arch medium --epochs 60 --lr 3e-4 --dropout 0.25 --weight_decay 1e-4 --run_name medium_best
!{base} --arch medium --epochs 60 --lr 1e-3 --dropout 0.25 --weight_decay 0.0 --run_name medium_highlr
!{base} --arch medium --epochs 60 --lr 1e-4 --dropout 0.5 --weight_decay 1e-3 --run_name medium_heavy_reg
!{base} --arch medium --epochs 60 --lr 3e-4 --dropout 0.25 --batch_size 256 --run_name medium_largebatch
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.3 --label_smoothing 0.1 --weight_decay 1e-4 --run_name resnet_best
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.3 --label_smoothing 0.0 --run_name resnet_no_smoothing
!{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.2 --weight_decay 1e-3 --run_name resnet_high_wd

[train] device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sdeli23 (sdeli23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 9me54vl6 (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/wandb/run-20260618_051550-9me54vl6
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run shallow_UNDERFIT
wandb: ⭐️ View project at https://wandb.ai/sdeli23-free-university-of-tbilisi-/fer2013-emotion
wandb: 🚀 View run at https://wandb.ai/sdeli23-free-university-of-tbilisi-/fer2013-emotion/runs/9me54vl6
Traceback (most recent call last):
  File "/content/src/train.py", line 243, in <module>
    main()
  File "/content/src/train.py", line 107, in main

In [ ]:
import os, sys, pandas as pd, numpy as np
from PIL import Image

os.makedirs('data', exist_ok=True)
os.makedirs('src', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
sys.path.insert(0, 'src')

# Step 1: Re-download images
os.system('kaggle datasets download -d msambare/fer2013 -p data/')
os.system('unzip -q data/fer2013.zip -d data/')

# Step 2: Build CSV
def folder_to_csv(split, output_path):
    emotion_map = {'angry':0,'disgust':1,'fear':2,'happy':3,'neutral':6,'sad':4,'surprise':5}
    rows = []
    for emotion_name, label in emotion_map.items():
        folder = f'data/{split}/{emotion_name}'
        if not os.path.exists(folder): continue
        for fname in os.listdir(folder):
            if not fname.endswith(('.jpg','.png')): continue
            img = Image.open(f'{folder}/{fname}').convert('L').resize((48,48))
            pixels = ' '.join(str(p) for p in np.array(img).flatten())
            rows.append({'emotion':label,'pixels':pixels,'Usage':'Training' if split=='train' else 'PublicTest'})
    pd.DataFrame(rows).to_csv(output_path, index=False)
    print(f'✅ {output_path} — {len(rows)} rows')

folder_to_csv('train','data/train.csv')
folder_to_csv('test','data/test.csv')

# Step 3: Train all experiments
base = 'python src/train.py --csv data/train.csv --wandb_project fer2013-emotion'

os.system(f'{base} --arch shallow --epochs 40 --lr 1e-4 --dropout 0.7 --run_name shallow_UNDERFIT')
os.system(f'{base} --arch medium --epochs 60 --lr 3e-4 --dropout 0.25 --weight_decay 1e-4 --run_name medium_best')
os.system(f'{base} --arch medium --epochs 60 --lr 1e-3 --dropout 0.25 --weight_decay 0.0 --run_name medium_highlr')
os.system(f'{base} --arch medium --epochs 60 --lr 1e-4 --dropout 0.5 --weight_decay 1e-3 --run_name medium_heavy_reg')
os.system(f'{base} --arch medium --epochs 60 --lr 3e-4 --dropout 0.25 --batch_size 256 --run_name medium_largebatch')
os.system(f'{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.3 --label_smoothing 0.1 --weight_decay 1e-4 --run_name resnet_best')
os.system(f'{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.3 --label_smoothing 0.0 --run_name resnet_no_smoothing')
os.system(f'{base} --arch resnet --epochs 80 --lr 1e-3 --dropout 0.2 --weight_decay 1e-3 --run_name resnet_high_wd')

print('🎉 ALL DONE!')

✅ data/train.csv — 28709 rows
✅ data/test.csv — 7178 rows
